In [1]:
!wget -O banglaclip_model_epoch_10.pth https://huggingface.co/Mansuba/BanglaCLIP13/resolve/main/banglaclip_model_epoch_10.pth



--2025-01-25 09:39:22--  https://huggingface.co/Mansuba/BanglaCLIP13/resolve/main/banglaclip_model_epoch_10.pth
Resolving huggingface.co (huggingface.co)... 3.165.160.61, 3.165.160.12, 3.165.160.59, ...
Connecting to huggingface.co (huggingface.co)|3.165.160.61|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/2a/ad/2aadee49f0f0c86b7b267a72f0937b06e9c98f694b60f9d09a1698caad62340d/f25c01d0773579e603903fefc52f721337cf92cbcbfb4ab6e48d2c858c8cbc3f?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27banglaclip_model_epoch_10.pth%3B+filename%3D%22banglaclip_model_epoch_10.pth%22%3B&Expires=1737801562&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczNzgwMTU2Mn19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzJhL2FkLzJhYWRlZTQ5ZjBmMGM4NmI3YjI2N2E3MmYwOTM3YjA2ZTljOThmNjk0YjYwZjlkMDlhMTY5OGNhYWQ2MjM0MGQvZjI1YzAxZDA3NzM1NzllNjAzOTAzZmVmYzUyZjcyMTMzN2NmOTJjYmNiZmI0YWI2ZTQ

In [2]:
!pip install huggingface_hub

In [3]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
The token `eswdgdhsdjcjksdjswo` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-a

In [6]:
import torch
import torch.nn as nn
import torch.quantization
import os

def quantize_model(model_path):
    """
    Quantize a PyTorch model loaded from a checkpoint
    """
    # 1. Load the checkpoint
    checkpoint = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)

    # 2. Extract model state dict
    # Modify this based on your exact checkpoint structure
    state_dict = checkpoint.get('model_state_dict', checkpoint)

    # 3. Reconstruct the model (REPLACE with your actual model class)
    class BanglaClipModel(nn.Module):
        def __init__(self):
            super(BanglaClipModel, self).__init__()
            # Define your actual model architecture here
            # This is a critical step - you must match your original model structure
            self.model = nn.Sequential(
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.Linear(256, 128)
            )

        def forward(self, x):
            return self.model(x)

    # 4. Create model instance and load state dict
    model = BanglaClipModel()

    # 5. Handle potential key mismatches
    new_state_dict = {}
    for k, v in state_dict.items():
        # Remove module. prefix if present
        if k.startswith('module.'):
            k = k[7:]
        # Remove model. prefix if present
        if k.startswith('model.'):
            k = k[6:]
        new_state_dict[k] = v

    # 6. Load state dict with partial matching
    model.load_state_dict(new_state_dict, strict=False)

    # 7. Prepare for quantization
    model.eval()
    model.qconfig = torch.quantization.get_default_qconfig('fbgemm')

    # 8. Prepare quantization
    quantized_model = torch.quantization.prepare(model)

    # 9. Calibration (use dummy input)
    dummy_input = torch.randn(1, 512)  # Adjust to your model's input shape
    quantized_model(dummy_input)

    # 10. Convert to quantized model
    quantized_model = torch.quantization.convert(quantized_model)

    # 11. Save quantized model
    quantized_model_path = model_path.replace('.pth', '_quantized.pth')
    torch.save({
        'model_state_dict': quantized_model.state_dict(),
        'quantized': True
    }, quantized_model_path)

    # 12. Size comparison
    original_size = os.path.getsize(model_path) / (1024 * 1024)
    quantized_size = os.path.getsize(quantized_model_path) / (1024 * 1024)

    print(f"Original model size: {original_size:.2f} MB")
    print(f"Quantized model size: {quantized_size:.2f} MB")
    print(f"Size reduction: {(1 - quantized_size/original_size)*100:.2f}%")

    return quantized_model

# Usage
model_path = 'banglaclip_model_epoch_10.pth'
quantized_model = quantize_model(model_path)

/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Original model size: 1538.67 MB
Quantized model size: 0.17 MB
Size reduction: 99.99%


In [7]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="/content/banglaclip_model_epoch_10_quantized.pth",
    path_in_repo="/content/banglaclip_model_epoch_10_quantized.pth",
    repo_id="Mansuba/Bangla_text_to_image_app",
    repo_type="space"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


banglaclip_model_epoch_10_quantized.pth:   0%|          | 0.00/177k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/spaces/Mansuba/Bangla_text_to_image_app/commit/1e5c7882a3b8efa4d4a051a86686d8b15df37a37', commit_message='Upload /content/banglaclip_model_epoch_10_quantized.pth with huggingface_hub', commit_description='', oid='1e5c7882a3b8efa4d4a051a86686d8b15df37a37', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/Mansuba/Bangla_text_to_image_app', endpoint='https://huggingface.co', repo_type='space', repo_id='Mansuba/Bangla_text_to_image_app'), pr_revision=None, pr_num=None)